# 1 · Transcrever (Whisper na GPU)

Transcreve os vídeos/áudios de `EditorVideos/<projeto>/entrada` e grava em `EditorVideos/<projeto>/transcricao`:

- `<nome>.json`: palavras com tempo (`[{text, start, end}]`), o formato que o HyperFrames importa para legendas palavra a palavra
- `<nome>.srt` e `<nome>.vtt`: legendas por frase (o FilmCraft importa `.srt`)

**Antes:** Ambiente de execução → Alterar tipo → **GPU T4**. Depois: Ambiente de execução → Executar tudo.
O andamento aparece no Painel, aba Colab.

In [ ]:
#@title 1. Configuração
PROJETO = "HyperFrames"  #@param ["HyperFrames", "FilmCraft"]
ARQUIVO = ""  #@param {type:"string"}
#@markdown Vazio = todos da entrada que ainda não têm transcrição.
IDIOMA = "pt"  #@param {type:"string"}
MODELO = "large-v3-turbo"  #@param ["large-v3-turbo", "large-v3", "medium", "small"]
REFAZER = False  #@param {type:"boolean"}

# Pelo Colab CLI os valores chegam por variável de ambiente (colab exec --env).
import os
PROJETO = os.environ.get('PROJETO', PROJETO)
ARQUIVO = os.environ.get('ARQUIVO', ARQUIVO)
REFAZER = os.environ.get('REFAZER', str(REFAZER)).lower() in ('1', 'true', 'sim')

In [ ]:
#@title 2. Drive e status para o Painel
from google.colab import drive
import json, os, time, threading, subprocess, collections
from datetime import datetime

RAIZ = '/content/drive/MyDrive/EditorVideos'
STATUS_DIR = f'{RAIZ}/_colab/status'

# A pasta EditorVideos vem do PC (Drive para computador). Se a montagem do Colab estiver velha ela
# não aparece; remonta e, se ainda faltar, para aqui em vez de criar uma "EditorVideos (1)" vazia.
# Pelo Colab CLI (Painel/colab-cli) o Drive já vem montado por `colab drivemount`: drive.mount() aqui
# ficaria esperando um navegador que não existe.
# Tarefas que não precisam do Drive (ex.: a voz pelo CLI) mandam EDITORVIDEOS_SEM_DRIVE=1: o status fica na VM
# e o Painel acompanha pelo log do CLI.
SEM_DRIVE = os.environ.get('EDITORVIDEOS_SEM_DRIVE') == '1'
if SEM_DRIVE:
    STATUS_DIR = '/content/_colab/status'
    print('Sem Drive nesta tarefa: o andamento vai pelo log do Colab CLI.')
elif os.environ.get('EDITORVIDEOS_CLI') and os.path.isdir('/content/drive/MyDrive'):
    print('Drive montado pelo Colab CLI.')
else:
    drive.mount('/content/drive', force_remount=True)
    if not os.path.isdir(f'{RAIZ}/HyperFrames'):
        time.sleep(10)
        drive.flush_and_unmount()
        drive.mount('/content/drive', force_remount=True)
if not SEM_DRIVE and not os.path.isdir(f'{RAIZ}/HyperFrames'):
    raise SystemExit('EditorVideos/HyperFrames não está visível neste Drive. Confira se o Colab está na mesma conta '
                     'do Drive do PC e se a sincronização terminou; depois Ambiente de execução → Desconectar e excluir.')

class Status:
    """Grava o andamento em _colab/status/<servico>.json. Batimento a cada 15 s enquanto roda."""
    def __init__(self, servico, notebook):
        os.makedirs(STATUS_DIR, exist_ok=True)
        self.arq = f'{STATUS_DIR}/{servico}.json'
        self.d = dict(servico=servico, notebook=notebook, estado='rodando', etapa='iniciando',
                      progresso=None, mensagem='', inicio=int(time.time() * 1000))
        self.log = collections.deque(maxlen=40)
        self._trava = threading.Lock()
        self._ultimo = 0
        self._vivo = True
        threading.Thread(target=self._batimento, daemon=True).start()
        self.gravar()

    def _gpu(self):
        try:
            out = subprocess.run(['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total',
                                  '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=5).stdout
            nome, uso, usada, total = [x.strip() for x in out.strip().splitlines()[0].split(',')]
            return dict(nome=nome, uso=int(uso), memoria_usada_mb=int(usada), memoria_total_mb=int(total))
        except Exception:
            return None

    def _ram(self):
        try:
            m = {}
            for linha in open('/proc/meminfo'):
                k, v = linha.split(':')
                m[k] = int(v.split()[0])
            return dict(usada_mb=(m['MemTotal'] - m['MemAvailable']) // 1024, total_mb=m['MemTotal'] // 1024)
        except Exception:
            return None

    def gravar(self):
        with self._trava:
            self.d.update(log=list(self.log), gpu=self._gpu(), ram=self._ram(), atualizado=int(time.time() * 1000))
            tmp = self.arq + '.tmp'
            with open(tmp, 'w') as f:
                json.dump(self.d, f, ensure_ascii=False)
            os.replace(tmp, self.arq)
            self._ultimo = time.time()

    def _batimento(self):
        while self._vivo:
            time.sleep(15)
            if self._vivo:
                self.gravar()

    def __call__(self, etapa=None, progresso=None, mensagem=None, linha=None):
        mudou = etapa is not None and etapa != self.d['etapa']
        if etapa is not None: self.d['etapa'] = etapa
        if progresso is not None: self.d['progresso'] = round(progresso, 1)
        if mensagem is not None: self.d['mensagem'] = mensagem
        if linha:
            self.log.append(f'{datetime.now():%H:%M:%S} {linha}')
            print(linha)
        # Não martela o Drive: no máximo uma gravação a cada 3 s, salvo mudança de etapa.
        if mudou or linha or time.time() - self._ultimo > 3:
            self.gravar()

    def fim(self, estado='concluido', mensagem=''):
        self._vivo = False
        self.d.update(estado=estado, mensagem=mensagem, etapa='fim' if estado == 'concluido' else self.d['etapa'])
        if estado == 'concluido': self.d['progresso'] = 100
        self.gravar()

print('Drive montado. Status em', STATUS_DIR)

In [ ]:
#@title 3. Instalar o faster-whisper
!pip -q install faster-whisper==1.1.1 nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

# O ctranslate2 procura o cuBLAS/cuDNN do CUDA 12; o Colab pode vir com outro CUDA. Carrega os do pip
# antes do modelo: o dlopen dele acha as bibliotecas já na memória. Repete porque umas dependem das outras.
import ctypes, glob, site
faltam = [l for d in site.getsitepackages() for pasta in ('cublas', 'cudnn')
          for l in sorted(glob.glob(f'{d}/nvidia/{pasta}/lib/lib*.so*'))]
for _ in range(4):
    pendentes = []
    for l in faltam:
        try:
            ctypes.CDLL(l, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            pendentes.append(l)
    faltam = pendentes
print('CUDA 12 carregado' if not faltam else f'não carregou: {faltam}')

In [ ]:
#@title 4. Transcrever
import glob, pathlib
import ctranslate2
from faster_whisper import WhisperModel

BASE = f'{RAIZ}/{PROJETO}'
ENTRADA, SAIDA = f'{BASE}/entrada', f'{BASE}/transcricao'
os.makedirs(SAIDA, exist_ok=True)
EXT = ('.mp4', '.mov', '.mkv', '.webm', '.m4v', '.mp3', '.m4a', '.wav', '.aac', '.ogg', '.opus')

# Decodifica com o ffmpeg do Colab (16 kHz mono): o PyAV novo do Colab quebra o decode do faster-whisper 1.1.1.
import numpy as np
def ler_audio(arq):
    pcm = subprocess.run(['ffmpeg', '-nostdin', '-v', 'error', '-i', arq, '-vn', '-ac', '1', '-ar', '16000', '-f', 's16le', '-'],
                         capture_output=True, check=True).stdout
    return np.frombuffer(pcm, np.int16).astype(np.float32) / 32768.0

def tempo(s, sep=','):
    ms = int(round(s * 1000))
    return f'{ms // 3600000:02}:{ms // 60000 % 60:02}:{ms // 1000 % 60:02}{sep}{ms % 1000:03}'

if ARQUIVO:
    arquivos = [f'{ENTRADA}/{ARQUIVO}']
else:
    arquivos = [p for p in sorted(glob.glob(f'{ENTRADA}/*')) if p.lower().endswith(EXT)
                and (REFAZER or not os.path.exists(f'{SAIDA}/{pathlib.Path(p).stem}.json'))]

st = Status('transcricao', '1_transcrever')
try:
    if not arquivos:
        st.fim('concluido', f'Nada novo para transcrever em {PROJETO}/entrada')
    else:
        gpu = ctranslate2.get_cuda_device_count() > 0
        st('carregando modelo', 0, f'{MODELO} · {"GPU" if gpu else "CPU (lento!)"}', f'{len(arquivos)} arquivo(s) para transcrever')
        modelo = WhisperModel(MODELO, device='cuda' if gpu else 'cpu', compute_type='float16' if gpu else 'int8')
        for i, arq in enumerate(arquivos):
            nome = pathlib.Path(arq).stem
            st(f'transcrevendo {i + 1}/{len(arquivos)}', i / len(arquivos) * 100, pathlib.Path(arq).name, f'início: {pathlib.Path(arq).name}')
            segs, info = modelo.transcribe(ler_audio(arq), language=IDIOMA or None, word_timestamps=True, vad_filter=True, beam_size=5)
            palavras, frases = [], []
            for s in segs:
                frases.append((s.start, s.end, s.text.strip()))
                palavras += [dict(text=w.word.strip(), start=round(w.start, 3), end=round(w.end, 3)) for w in (s.words or []) if w.word.strip()]
                st(progresso=(i + min(1, s.end / max(info.duration, 1e-6))) / len(arquivos) * 100)
            with open(f'{SAIDA}/{nome}.json', 'w') as f:
                json.dump(palavras, f, ensure_ascii=False, indent=1)
            with open(f'{SAIDA}/{nome}.srt', 'w') as f:
                f.write('\n'.join(f'{n}\n{tempo(a)} --> {tempo(b)}\n{t}\n' for n, (a, b, t) in enumerate(frases, 1)))
            with open(f'{SAIDA}/{nome}.vtt', 'w') as f:
                f.write('WEBVTT\n\n' + '\n'.join(f'{tempo(a, ".")} --> {tempo(b, ".")}\n{t}\n' for a, b, t in frases))
            st(linha=f'ok: {nome} ({len(palavras)} palavras, {info.duration / 60:.1f} min, idioma {info.language})')
        st.fim('concluido', f'{len(arquivos)} arquivo(s) em {PROJETO}/transcricao')
except Exception as e:
    st(linha=f'ERRO: {e}')
    st.fim('erro', str(e))
    raise